# Import libraries

In [9]:
import os
from pathlib import Path

import gradio as gr
import pandas as pd

from dotenv import load_dotenv

from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel

# Load .env

In [10]:
# Load environment variables from .env
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY not found. "
        "Please add it to your .env file."
    )

print("✅ Environment variables loaded successfully.")

✅ Environment variables loaded successfully.


# Configuration

In [11]:
# Project configuration

APP_NAME = "AI ML Code Agent"
AUTHOR_NAME = "Ajim Mujawar"

# Current Gemini model specified by the API error
MODEL_ID = "gemini/gemini-3.8-flash"

MAX_AGENT_STEPS = 12

print(f"👤 Developer : {AUTHOR_NAME}")
print(f"🤖 Model     : {MODEL_ID}")

👤 Developer : Ajim Mujawar
🤖 Model     : gemini/gemini-3.8-flash


# Create a Gimini Model

In [12]:
model = LiteLLMModel(
    model_id=MODEL_ID,
    api_key=GOOGLE_API_KEY,
    temperature=0.2,
    max_tokens=8192
)

print("✅ Gemini model initialized successfully.")

✅ Gemini model initialized successfully.


# Create The Agent

In [29]:
ml_code_agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool()
    ],
    
    additional_authorized_imports=[
        "pandas",
        "numpy",
        "sklearn",
        "sklearn.model_selection",
        "sklearn.preprocessing",
        "sklearn.metrics",
        "sklearn.linear_model",
        "sklearn.ensemble",
        "sklearn.tree",
        "sklearn.pipeline",
        "sklearn.compose",
        "json",
        "os"
    ],
    
    model=model,
    
    max_steps=MAX_AGENT_STEPS,
    
    verbosity_level=1
)

print("✅ ML Code Agent created successfully.")

✅ ML Code Agent created successfully.


# System Prompts

In [14]:
SYSTEM_INSTRUCTIONS = """
You are an expert Machine Learning Engineering Agent.

Your job is to solve machine learning and data science tasks
using clean, production-quality Python.

Follow these rules:

1. Understand the user's requirement first.
2. If a CSV dataset is provided, inspect its structure.
3. Identify numerical and categorical features.
4. Identify the target column when possible.
5. Handle missing values appropriately.
6. Perform train/test splitting correctly.
7. Apply preprocessing when required.
8. Prevent data leakage.
9. Select suitable ML algorithms.
10. Train and evaluate models.
11. Use appropriate metrics.
12. Compare models when useful.
13. Explain important decisions.
14. Generate clean, readable Python code.
15. Prefer scikit-learn pipelines when appropriate.
16. Do not invent dataset columns.
17. Do not expose API keys or secrets.
18. If information is missing, clearly state the assumption.

When generating code:

- Use Python.
- Use pandas and scikit-learn where appropriate.
- Add comments.
- Make the code reproducible.
- Use random_state where applicable.
- Include evaluation metrics.
- Include final recommendations based on measured results.

Return the final answer in this structure:

## Problem Understanding

## Dataset Analysis

## Approach

## Complete Python Code

## Model Evaluation

## Key Findings

## Next Steps
"""

# CSV Analyzer

In [15]:
def analyze_csv(file_path):
    """
    Analyze an uploaded CSV file and return
    useful dataset information for the agent.
    """
    
    if not file_path:
        return "No CSV file uploaded."

    try:
        df = pd.read_csv(file_path)

        analysis = f"""
DATASET INFORMATION

Rows: {df.shape[0]}
Columns: {df.shape[1]}

Column Names:
{list(df.columns)}

Data Types:
{df.dtypes.to_string()}

Missing Values:
{df.isnull().sum().to_string()}

Duplicate Rows:
{df.duplicated().sum()}

Sample Data:
{df.head(5).to_string(index=False)}

Statistical Summary:
{df.describe(include="all").to_string()}
"""

        return analysis

    except Exception as e:
        return f"CSV analysis error: {str(e)}"

# Main Agent function

In [16]:
import time


def run_agent_with_retry(prompt, max_retries=4):
    """
    Run the ML agent with automatic retry
    for temporary Gemini availability errors.
    """

    for attempt in range(1, max_retries + 1):

        try:
            response = ml_code_agent.run(prompt)
            return str(response)

        except Exception as e:

            error_message = str(e)

            # Retry temporary service errors
            if "503" in error_message or "ServiceUnavailable" in error_message:

                if attempt < max_retries:

                    wait_time = 2 ** attempt

                    print(
                        f"⚠️ Gemini temporarily unavailable. "
                        f"Retry {attempt}/{max_retries} "
                        f"in {wait_time}s..."
                    )

                    time.sleep(wait_time)

                else:
                    return (
                        "⚠️ Gemini is currently experiencing "
                        "high demand.\n\n"
                        "Please try again after a few minutes."
                    )

            else:
                return f"❌ Agent Error\n\n{error_message}"

# Example Prompt

In [17]:
EXAMPLE_PROMPTS = [
    "Analyze this dataset and identify the target column.",
    
    "Build a complete classification pipeline for this dataset.",
    
    "Compare Logistic Regression, Random Forest and Gradient Boosting.",
    
    "Perform EDA and explain the important patterns.",
    
    "Build a regression model and compare at least three algorithms.",
    
    "Handle missing values and categorical features properly."
]

# Professional Gradio CSS

In [18]:
custom_css = """
:root {
    --radius: 16px;
}

.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
}

#hero {
    text-align: center;
    padding: 25px;
    border-radius: 20px;
    margin-bottom: 20px;
}

#hero h1 {
    font-size: 38px;
    margin-bottom: 8px;
}

#hero p {
    font-size: 17px;
}

#developer {
    text-align: center;
    margin-top: 15px;
    padding: 12px;
    font-size: 14px;
}

#generate-btn {
    min-height: 50px;
    font-size: 17px;
    font-weight: 600;
}

textarea {
    font-size: 15px !important;
}
"""

# Professional Gradio UI

In [22]:
import pandas as pd

def analyze_csv(csv_path):

    if csv_path is None:
        return "❌ Please upload a CSV file."

    try:
        df = pd.read_csv(csv_path)

        result = "### 📊 Dataset Overview\n\n"
        result += f"**Rows:** {df.shape[0]}\n\n"
        result += f"**Columns:** {df.shape[1]}\n\n"

        result += "### 🧾 Columns\n\n"
        result += ", ".join(df.columns.astype(str).tolist())
        result += "\n\n"

        result += "### 🔢 Data Types\n\n"
        result += "```\n"
        result += df.dtypes.to_string()
        result += "\n```\n\n"

        result += "### ❓ Missing Values\n\n"
        result += "```\n"
        result += df.isnull().sum().to_string()
        result += "\n```\n\n"

        result += "### 👀 Preview\n\n"
        result += "```\n"
        result += df.head().to_string()
        result += "\n```"

        return result

    except Exception as e:
        return "❌ Error reading CSV: " + str(e)

In [25]:
def generate_ml_solution(prompt, csv_path):

    if csv_path is None:
        return "❌ Please upload a CSV dataset first."

    if not prompt or not prompt.strip():
        return "❌ Please describe your ML task."

    try:
        df = pd.read_csv(csv_path)

        dataset_info = (
            "Dataset shape: " + str(df.shape) + "\n\n"
            "Columns: " + str(df.columns.tolist()) + "\n\n"
            "Data types:\n" + df.dtypes.to_string() + "\n\n"
            "Missing values:\n" + df.isnull().sum().to_string() + "\n\n"
            "First 5 rows:\n" + df.head().to_string()
        )

        task = (
            "You are an expert Machine Learning Engineer.\n\n"
            "Dataset information:\n"
            + dataset_info
            + "\n\nUser's ML task:\n"
            + prompt
            + "\n\n"
            "Provide a practical Machine Learning solution.\n"
            "Include:\n"
            "1. Problem understanding\n"
            "2. Data preprocessing\n"
            "3. Feature selection\n"
            "4. Recommended algorithms\n"
            "5. Training approach\n"
            "6. Evaluation metrics\n"
            "7. Complete Python code\n"
            "8. Explanation\n\n"
            "Do not invent column names."
        )

        response = agent.run(task)

        return str(response)

    except Exception as e:
        return "❌ Error while generating solution: " + str(e)

In [26]:
with gr.Blocks(
    title="AI ML Code Agent | Ajim Mujawar",
    theme=gr.themes.Soft(),
    css=custom_css
) as demo:

    gr.HTML(
        """
        <div id="hero">
            <h1>🤖 AI ML Code Agent</h1>
            <p>
                Intelligent Machine Learning assistant powered by
                <b>SmolAgents + Gemini + Python</b>
            </p>
            <p>
                Upload a CSV, describe your ML task, and let the agent
                analyze, reason, and generate the solution.
            </p>
        </div>
        """
    )

    with gr.Row():

        # LEFT SIDE
        with gr.Column(scale=1):

            gr.Markdown("## 📂 Dataset")

            csv_input = gr.File(
                label="Upload CSV Dataset",
                file_types=[".csv"],
                type="filepath"
            )

            dataset_preview = gr.Textbox(
                label="Dataset Information",
                lines=12,
                interactive=False
            )

            analyze_btn = gr.Button(
                "🔍 Analyze Dataset"
            )

        # RIGHT SIDE
        with gr.Column(scale=2):

            gr.Markdown("## 💬 ML Task")

            prompt_input = gr.Textbox(
                label="Describe your ML task",
                placeholder=(
                    "Example: Build a classification model, "
                    "compare algorithms and show evaluation metrics..."
                ),
                lines=7
            )

            gr.Examples(
                examples=EXAMPLE_PROMPTS,
                inputs=prompt_input,
                label="💡 Example Tasks"
            )

            generate_btn = gr.Button(
                "🚀 Generate ML Solution",
                variant="primary",
                elem_id="generate-btn"
            )

    gr.Markdown("---")

    gr.Markdown("## 🧠 AI Generated Solution")

    output = gr.Markdown(
        value="Your generated ML solution will appear here..."
    )

    analyze_btn.click(
        fn=analyze_csv,
        inputs=csv_input,
        outputs=dataset_preview
    )

    generate_btn.click(
        fn=generate_ml_solution,
        inputs=[
            prompt_input,
            csv_input
        ],
        outputs=output
    )

    gr.HTML(
        """
        <div id="developer">
            🚀 Built by <b>Ajim Mujawar</b>
            <br>
            AI / ML Engineer | Python | Machine Learning | Generative AI
        </div>
        """
    )

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14024\398424279.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [27]:
print(generate_ml_solution)

<function generate_ml_solution at 0x0000027DA0619300>


# Launch

In [28]:
demo.launch(
    inbrowser=True,
    share=False
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [1]:
# Cell 1
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("API key loaded:", bool(GOOGLE_API_KEY))

API key loaded: True


In [2]:
from smolagents import LiteLLMModel

MODEL_ID = "gemini/gemini-3.8-flash"

model = LiteLLMModel(
    model_id=MODEL_ID,
    api_key=GOOGLE_API_KEY,
    temperature=0.2,
    max_tokens=2048
)

print("Model initialized:", MODEL_ID)

c:\Users\ADMIN\anaconda3\envs\csv_agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model initialized: gemini/gemini-3.8-flash


In [4]:
from smolagents import ChatMessage

messages = [
    ChatMessage(
        role="user",
        content=[
            {
                "type": "text",
                "text": "Reply with exactly: Gemini is working."
            }
        ]
    )
]

response = model.generate(messages)

print(response.content)

15:37:51 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1109 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.8-flash) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.


Gemini is working.


In [7]:
from smolagents import ChatMessage

test = model.generate([
    ChatMessage(
        role="user",
        content="Reply with exactly: Gemini is working."
    )
])

print(test.content)

15:40:30 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1109 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.8-flash) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.


Gemini is working.
